In [1]:
import math

In [5]:
import math

class Value:                             #a number which remembers how it was made, Python can add numbers but doesn't know how to add Values 

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data                 # the number itself
        self.grad = 0.0                  # filled in later, by backward()
        self._backward = lambda: None
        self._prev = set(_children)      # which Values this one came from
        self._op = _op                   # which operation made it
        self.label = label               # a name, for drawing graphs

    def __repr__(self):                 #data output readibility
        return f"Value(data={self.data}, grad={self.grad})"

    # addition 
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)   # wrap plain numbers but if it's already a Value take it 
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad  += out.grad       # '+' passes the gradient through unchanged...
            other.grad += out.grad       # ...to BOTH inputs.  += not =, always.
        out._backward = _backward        # store the function, don't call it

        return out

    # multiplication
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad  += other.data * out.grad   # '*' SWAPS: each input gets
            other.grad += self.data  * out.grad   # the OTHER one's value
        out._backward = _backward

        return out

    # power, with a plain-number exponent
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "exponent must be a plain number"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out

    # tanh squashing 
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad    # the tanh gate: 1 - tanh(x)^2
        out._backward = _backward

        return out

    # built out of the ones above, no new backward passes 
    def __neg__(self):            return self * -1
    def __sub__(self, other):     return self + (-other)
    def __radd__(self, other):    return self + other      # so 2 + x works
    def __rmul__(self, other):    return self * other      # so 2 * x works
    def __truediv__(self, other): return self * other**-1

    # the backward pass 
    def backward(self):
        topo = []                 # is the ordering we are building, every node, parents always after children (right order calculation)
        visited = set()           # never process the same node twice 

        def build_topo(v):
            if v not in visited:    #if already node visited then skip 
                visited.add(v)      #else mark it
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0           # seed: the output's sensitivity to itself is 1
        for v in reversed(topo):  # walk parents first, so each node's grad is
            v._backward()         # already filled in before it hands gradient down

        
    # ---------- relu, the other squashing function ----------
    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def __rsub__(self, other):     return (-self) + other      # so 2 - x works
    def __rtruediv__(self, other): return (self**-1) * other    # so 10 / x works

In [ ]:
# forward pass 
x1, x2       = Value(1.0), Value(0.5)

w11, w21, b1 = Value(0.5),  Value(0.4),  Value(0.1)
w12, w22, b2 = Value(-0.3), Value(0.8),  Value(0.2)
v1, v2, c    = Value(0.6),  Value(-0.5), Value(0.1)

# computed 
z1 = x1*w11 + x2*w21 + b1;  h1 = z1.tanh()
z2 = x1*w12 + x2*w22 + b2;  h2 = z2.tanh()
y    = h1*v1 + h2*v2 + c
loss = (y - 1.0)**2

print(f"z1={z1.data:.4f}  h1={h1.data:.4f}   z2={z2.data:.4f}  h2={h2.data:.4f}")
print(f"y={y.data:.4f}   loss={loss.data:.4f}")

In [4]:
# backward pass
loss.backward()

for name, p in [('w11',w11), ('w21',w21), ('b1',b1),
                ('w12',w12), ('w22',w22), ('b2',b2),
                ('v1',v1),   ('v2',v2),   ('c',c)]:
    print(f"{name:<4} {p.grad:+.6f}")                   #formating 

print(f"\nx1   {x1.grad:+.6f}   <- two paths, summed")
print(f"tanh gate, neuron 1: {1 - h1.data**2:.6f}")
print(f"tanh gate, neuron 2: {1 - h2.data**2:.6f}")

NameError: name 'loss' is not defined

In [ ]:
# update
params = [w11, w21, b1, w12, w22, b2, v1, v2, c]

for step in range(50):

    # --- forward: rebuild the graph with the CURRENT parameter values
    z1 = x1*w11 + x2*w21 + b1;  h1 = z1.tanh()
    z2 = x1*w12 + x2*w22 + b2;  h2 = z2.tanh()
    y    = h1*v1 + h2*v2 + c
    loss = (y - 1.0)**2

    # --- reset gradients from last step
    for p in params:
        p.grad = 0.0                    #really important step 

    # --- backward
    loss.backward()

    # --- nudge each parameter downhill
    for p in params:
        p.data -= 0.1 * p.grad

    print(f"step {step:2d}   loss {loss.data:.6f}   y {y.data:.4f}")